In [ ]:
#| default_exp core

# API

> API for ipykernel-helper

In [ ]:
from fasthtml.common import *
from pyskills import resolve
from fastcore.meta import delegates
from fastcore.utils import patch,patch_to
from fastcore.docments import sig_source,DocmentText
from fastcore.net import HTTP404NotFoundError
from fastcore.xtras import truncstr
from fastcore.aio import maybe_await,enable_async_magics
from types import ModuleType, FunctionType, MethodType, BuiltinFunctionType
from functools import cmp_to_key,partial
from textwrap import dedent
from cloudscraper import create_scraper
from fastcore.funccall import *
from toolslm.xml import *
import ipyfuncs
from urllib.parse import urlparse, urljoin

import re,os,html2text,base64,inspect

from IPython.core.interactiveshell import InteractiveShell

from IPython.core.display import DisplayObject
from IPython.display import display,Markdown,HTML
from IPython.core.oinspect import Inspector
from IPython.core.displayhook import DisplayHook
from ipykernel.displayhook import ZMQShellDisplayHook
from IPython.core.formatters import DisplayFormatter

In [ ]:
from fastcore.test import *
from fastcore.utils import *

from pprint import pprint

## InteractiveShell helpers

In [ ]:
@patch
def xpush(self:InteractiveShell, interactive=False, **kw):
    "Like `push`, but with kwargs"
    self.push(kw, interactive=interactive)

In [ ]:
ipy = get_ipython()

In [ ]:
ipy.push(dict(a=2))
a

2

In [ ]:
# ipykernel_helper version uses `**kwargs`
ipy.xpush(a=3)
a

3

The main benefits of using `ipy.push(dict(a=2))` over directly executing code are:

1. **Bulk variable assignment** - You can set multiple variables at once with a single command
2. **Programmatic variable injection** - It provides a way to inject variables into the namespace from another context or function
3. **No execution history** - Variables are added without creating an entry in the execution history
4. **No side effects** - It's a "pure" namespace modification without executing any code that might have side effects

There are several interesting functions in the IPython interpreter object that are useful for notebook development and interactive computing:

1. **`reset`/`reset_selective`** - Clear variables from the namespace (either all or selectively)
2. **`run_cell`/`run_cell_async`** - Execute code in a cell programmatically
3. **`set_next_input`** - Programmatically set the content of the next cell
4. **`system`/`system_raw`/`system_piped`** - Execute shell commands with different output handling
5. **`run_line_magic`/`run_cell_magic`** - Execute IPython magics programmatically
6. **`set_custom_exc`** - Set custom exception handlers

## Displaying MIME data

In [ ]:
def transient(data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    display({f'{mimetype}/{subtype}': data}, raw=True, metadata=meta, transient=kw, update=update)

In [ ]:
transient('hi there', foo='bar')

hi there

In [ ]:
transient('*hi* **there**', subtype='markdown')

*hi* **there**

In [ ]:
def run_cmd(cmd, data='', meta=None, update=False, **kw):
    transient(data, meta=meta, update=update, cmd=cmd, **kw)

## read_url et al

In [ ]:
def _absolutify_imgs(md, base_url):
    def fix(m):
        alt,img_url = m.group(1),m.group(2)
        if not img_url.startswith('http'): img_url = urljoin(base_url, img_url)
        alt = alt.replace('\\','')
        return f'![{alt}]({img_url})'
    return re.sub(r'!\[(.*?)\]\((.*?)\)', fix, md)

In [ ]:
_md = 'An alt text with escape chars\n![\[Uncaptioned image\]](https://www.example.org)'

Running the following will crash solveit:

In [ ]:
# DON'T RUN!
# md_bundle = { 'text/markdown': md}
# ipy.display_pub.publish(data=md_bundle)

This happens because `EscapeSequence` isn't handled correctly inside `FrankenRenderer.render_image` and raises an exception. We fix this by replacing the escape characters in the image alt-text:

In [ ]:
cts = _absolutify_imgs(_md, '')
md_bundle = { 'text/markdown': cts }
ipy.display_pub.publish(data=md_bundle)

An alt text with escape chars
![[Uncaptioned image]](https://www.example.org)

In [ ]:
def get_md(html, url='', mmode=None, ignore_links=False, ignore_images=False, mark_code=True):
    "Convert HTML to markdown with absolute image URLs and optional math mode"
    h = html2text.HTML2Text()
    h.body_width = 0
    h.ignore_links, h.ignore_images, h.mark_code = ignore_links, ignore_images, mark_code
    res = _absolutify_imgs(h.handle(str(html)), url)
    if mmode == 'safe': res = res.replace(r'\\(',r'\(').replace(r'\\)',r'\)')
    return re.sub(r'\[code]\s*\n(.*?)\n\[/code]', lambda m: f'````\n{dedent(m.group(1))}\n````', res, flags=re.DOTALL).strip()

In [ ]:
def scrape_url(url): 
    o = create_scraper().get(url)
    if not o.encoding or o.encoding == 'ISO-8859-1': o.encoding = 'utf-8'
    return o

In [ ]:
#| eval: false
scrape_url('http://www.example.org').encoding

'utf-8'

In [ ]:
def _get_math_mode():
    v = os.getenv('USE_KATEX', '')
    if v.lower() in {'0', 'false', 'none', ''}: return None
    return 'dollar' if v.lower().startswith('d') else 'safe'

In [ ]:
def _aify_imgs(md): return re.sub(r'!\[(.*?)\]\((.*?)\)', r'![\1](\2#ai)', md)

In [ ]:
def _extract_section(soup, url, selector=None):
    "Extract a specific section from soup, or the whole thing"
    if selector: return '\n\n'.join(str(s) for s in soup.select(selector))
    parsed = urlparse(url)
    if not parsed.fragment: return str(soup)
    section = soup.find(id=parsed.fragment)
    if not section: return ''
    elements = [section]
    current = section.next_sibling
    while current:
        if hasattr(current, 'name') and current.name == section.name: break
        elements.append(current)
        current = current.next_sibling
    return ''.join(str(el) for el in elements)

In [ ]:
def _convert_math(soup, mode):
    for math in soup.find_all('math'):
        annot = math.find('annotation', {'encoding': 'application/x-tex'})
        if not annot: continue
        tex,display = annot.text.strip(), math.get('display') == 'block'
        if mode == 'dollar': wrap = f'$${tex}$$' if display else f'${tex}$'
        else: wrap = f'$${tex}$$' if display else fr'\({tex}\)'
        math.replace_with(wrap)

In [ ]:
def read_url(
    url:str, # URL to read
    as_md:bool=True, # Convert HTML to markdown
    extract_section:bool=True, # Extract section matching URL fragment or selector
    selector:str=None, # CSS selector to extract specific content
    ai_img:bool=False # Add #ai suffix to image URLs
):
    "Read url from web"
    from bs4 import BeautifulSoup
    o = scrape_url(url)
    ctype = (o.headers.get('content-type') or 'text/plain').split(';')[0]
    res = o.text
    if ctype == 'text/html':
        soup = BeautifulSoup(res, 'lxml')
        if ('#' in url and extract_section) or selector: soup = BeautifulSoup(_extract_section(soup, url, selector), 'lxml')
        mmode = _get_math_mode()
        if mmode: _convert_math(soup, mmode)
        base = soup.find('base')
        base_url = urljoin(url, base['href'] if base else '')
        res = get_md(soup, base_url, mmode) if as_md else str(soup)
    if ai_img: res = _aify_imgs(res)
    return res

In [ ]:
print(read_url('https://www.example.org'))

# Example Domain

This domain is for use in documentation examples without needing permission. Avoid use in operations.

[Learn more](https://iana.org/domains/example)


In [ ]:
print(read_url('https://www.example.org', as_md=False, selector='body'))

<html><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>


In [ ]:
print(read_url('https://caddyserver.com/docs/running#unit-files'))

### Unit Files

We provide two different systemd unit files that you can choose between, depending on your use case:

  * [**`caddy.service`**](https://github.com/caddyserver/dist/blob/master/init/caddy.service) if you configure Caddy with a [Caddyfile](/docs/caddyfile). If you prefer to use a different config adapter or a JSON config file, you may override the `ExecStart` and `ExecReload` commands.

  * [**`caddy-api.service`**](https://github.com/caddyserver/dist/blob/master/init/caddy-api.service) if you configure Caddy solely through its [API](/docs/api). This service uses the [`--resume`](/docs/command-line#caddy-run) option which will start Caddy using the `autosave.json` which is [persisted](/docs/json/admin/config/) by default.




They are very similar, but differ in the `ExecStart` and `ExecReload` commands to accommodate the workflows.

If you need to switch between the services, you should disable and stop the previous one before enabling and starting the other. For example

`html2text` removes new lines so we only use `get_md` on html content and static text/markdown content is returned as is with new lines preserved:

In [ ]:
print('\n'.join(read_url('https://fastht.ml/docs/llms.txt').splitlines()[:5]))

# FastHTML

> FastHTML is a python library which brings together Starlette, Uvicorn, HTMX, and fastcore's `FT` "FastTags" into a library for creating server-rendered hypermedia applications. The `FastHTML` class itself inherits from `Starlette`, and adds decorator-based routing with many additions, Beforeware, automatic `FT` to HTML rendering, and much more.

Things to remember when writing FastHTML apps:


`read_url` renders latex based on the solveit user's katex setting (`USE_KATEX`)

In [ ]:
os.environ['USE_KATEX']='dollar'
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#S3\.SS2\.SSS1')[:700])

####  3.2.1 Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension $d_{k}$, and values of dimension $d_{v}$. We compute the dot products of the query with all keys, divide each by $\sqrt{d_{k}}$, and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix $Q$. The keys and values are also packed together into matrices $K$ and $V$. We compute the matrix of outputs as:

| $$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}(\frac{QK^{T}}{\sqrt{d_{k}}})V$$ |  | (1)  
---|---|---|--- 


In [ ]:
os.environ['USE_KATEX']='1'
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#S3\.SS2\.SSS1')[:700])

####  3.2.1 Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension \(d_{k}\), and values of dimension \(d_{v}\). We compute the dot products of the query with all keys, divide each by \(\sqrt{d_{k}}\), and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix \(Q\). The keys and values are also packed together into matrices \(K\) and \(V\). We compute the matrix of outputs as:

| $$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}(\frac{QK^{T}}{\sqrt{d_{k}}})V$$ |  | (1)  
---|


Relative image paths are automatically corrected as well:

In [ ]:
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#Sx1')[:700])

## Attention Visualizations

![Refer to caption](https://arxiv.org/html/1706.03762v7/x1.png) Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making…more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

![Refer to caption](https://arxiv.org/html/1706.03762v7/x2.png)

![Refer to caption](https://arxiv.org/html/1706.03762v7/x3.png)

Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top


In [ ]:
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#Sx1',ai_img=True)[:700])

## Attention Visualizations

![Refer to caption](https://arxiv.org/html/1706.03762v7/x1.png#ai) Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making…more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

![Refer to caption](https://arxiv.org/html/1706.03762v7/x2.png#ai)

![Refer to caption](https://arxiv.org/html/1706.03762v7/x3.png#ai)

Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolu


## Other helpers

In [ ]:
def fix_editable_priority():
    import sys
    from importlib.machinery import PathFinder
    try: sys.meta_path.append(sys.meta_path.pop(sys.meta_path.index(PathFinder)))
    except ValueError: pass

In [ ]:
prose_types = { 'pandas.DataFrame', 'pandas.io.formats.style.Styler',
    'polars.dataframe.frame.DataFrame', 'polars.series.series.Series' }

def _type_mros(o): return [f'{c.__module__}.{c.__qualname__}' for c in type(o).__mro__]
def _is_prose(o): return any(isinstance(o, t) if isinstance(t, type) else t in _type_mros(o) for t in prose_types)

@patch
def format(self:DisplayFormatter, obj, include=None, exclude=None):
    data,meta = self._orig_format(obj, include=include, exclude=exclude)
    if 'text/html' in data and _is_prose(obj): meta.setdefault('text/html', {})['prose'] = True
    return data,meta


In [ ]:
class Foo:
    def _repr_markdown_(self): return "1. hi\n2. bye"
Foo()

<div class="prose" markdown="1">

1. hi
2. bye

</div>

In [ ]:
Markdown("1. hi\n2. bye")

<div class="prose" markdown="1">

1. hi
2. bye

</div>

In [ ]:
from pandas import DataFrame
import pandas.io.formats.style

In [ ]:
df = DataFrame({'a':[1,2,3], 'b':[4,5,6]})
df

,a,b
0,1,4
1,2,5
2,3,6


In [ ]:
df.style.set_properties(color='red'
    ).set_table_styles([dict(selector='th', props='font-style: italic')])

,a,b
0,1,4
1,2,5
2,3,6


In [ ]:
from IPython.display import TextDisplayObject,DisplayObject
from pathlib import PurePath

In [ ]:
@patch
def __repr__(self:DisplayObject):
    s = self.data or self.url or self.filename
    if s is None: return None
    if not isinstance(s, str): s = truncstr(str(s), 30)
    return f"{type(self).__name__}({s})"

`DisplayObject.__init__` treats a `data` string as a URL to download if it starts with `http`, or as a filename to load if a file of that name exists on disk. This is undocumented, and it trips up the text display types: `Markdown(s)` where `s` happens to start with `https://` tries to fetch `s` as a web page and errors. We drop that guesswork for `TextDisplayObject` (the base of `Markdown`, `HTML`, `Latex`, `Math`, and friends), so text is always treated as text. Passing `url=` or `filename=` explicitly still works, since `reload()` is unchanged.

In [ ]:
@patch
def __init__(self:TextDisplayObject, data=None, url=None, filename=None, metadata=None):
    "Like `DisplayObject.__init__`, but never auto-detects `data` as a URL/filename"
    if isinstance(data, (Path, PurePath)): data = str(data)
    self.url, self.filename, self.data = url, filename, data
    if metadata is not None: self.metadata = metadata
    elif self.metadata is None: self.metadata = {}
    self.reload()
    self._check_data()

In [ ]:
async def call_tool(func, kw):
    "Call `func(**kw)` with `coerce_inputs`"
    return await maybe_await(func(**coerce_inputs(func, kw)))

Handles async:

In [ ]:
async def async_add(a:int, b:int): return a+b
test_eq(await call_tool(async_add, {'a':2, 'b':3}), 5)

string → list coercion via `coerce_inputs`:

In [ ]:
coerce_inputs??

````python
def coerce_inputs(func, inputs):
    "Coerce inputs based on function type annotations, repairing common LLM container mistakes"
    hints = get_type_hints(func) if hasattr(func, '__annotations__') else {}
    sig, res = inspect.signature(func), {}
    for k,v in inputs.items():
        ann = hints.get(k)
        if v is None and (p := sig.parameters.get(k)) and p.default is not empty: continue
        if ann in custom_types: res[k] = ann(v); continue
        cont, arr = _ann_outer(ann, (dict,list,tuple,set)), _ann_outer(ann, (list,tuple,set))
        if cont and isinstance(v, str) and not _allows_str(ann):
            try: v = ast.literal_eval(v)
            except Exception: pass
        if arr and v == {}: v = []
        elif arr and isinstance(v, str) and not _allows_str(ann): v = [v]
        if isinstance(v, dict) and callable(ann): res[k] = ann(**v)
        else: res[k] = v
    return res
````

**File:** `~/aai-ws/toolslm/toolslm/funccall.py`; line: 206

In [ ]:
def join_items(xs:list[str], sep:str=','): return sep.join(xs)
    
test_eq(await call_tool(join_items, {'xs':"['a','b','c']", 'sep':'/'}), 'a/b/c')

## Extension

In [ ]:
def _lineno(obj):
    "Source line number of `obj`, or `None` if unavailable"
    try: return inspect.getsourcelines(obj)[1]
    except (TypeError, OSError): return None

def _src_target(obj):
    "Object whose file/source best represents `obj`: unwrap wrappers, properties, and partials; instances resolve to their type"
    if not hasattr(obj, '__signature__'): obj = getattr(obj, '__wrapped__', obj)
    obj = getattr(obj, 'fget', None) or getattr(obj, 'fset', None) or obj
    if isinstance(obj, partial): obj = obj.func
    if isinstance(obj, (type, ModuleType, FunctionType, MethodType, BuiltinFunctionType)): return obj
    return type(obj)

In [ ]:
@patch
def _get_info(self:Inspector, obj, oname='', formatter=None, info=None, detail_level=0, omit_sections=()):
    "Custom formatter for ? and ?? output"
    orig = self._orig__get_info(obj, oname=oname, formatter=formatter, info=info,
                                detail_level=detail_level, omit_sections=omit_sections)
    try:
        tgt = _src_target(obj)
        oinfo = self.info(obj, oname=oname, info=info, detail_level=0)
        src = self.info(tgt, detail_level=2).get('source') if detail_level else None
        out = [f"````python\n{dedent(src) if src else DocmentText(obj)}\n````"]
        if not detail_level and tgt is type(obj) and (c:=oinfo.get('string_form')):
            out.append(f"**String form:** `{truncstr(c.splitlines()[0], 100)}`")
        finfo = oinfo if tgt is obj else self.info(tgt)
        if c:=finfo.get('file'):
            ln = _lineno(tgt)
            out.append(f"**File:** `{c}`" + (f"; line: {ln}" if ln else ""))
        if not detail_level and (c:=oinfo.get('type_name')): out.append(f"**Type:** {c}")
        return {'text/markdown': '\n\n'.join(out), 'text/html': '', 'text/plain': orig['text/plain']}
    except Exception: return orig

In [ ]:
flexiclass?

````python
def flexiclass(
    cls, # The class to convert
)->dataclass:
    "Convert `cls` into a `dataclass` like `make_nullable`. Converts in place and also returns the result."
````

**File:** `~/aai-ws/fastcore/fastcore/xtras.py`; line: 1003

**Type:** function

In [ ]:
flexiclass??

````python
def flexiclass(
        cls # The class to convert
    ) -> dataclass:
    "Convert `cls` into a `dataclass` like `make_nullable`. Converts in place and also returns the result."
    if is_dataclass(cls): return make_nullable(cls)
    for k,v in get_annotations_ex(cls)[0].items():
        if not hasattr(cls,k) or getattr(cls,k) is MISSING:
            setattr(cls, k, field(default=UNSET))
    return dataclass(cls, init=True, repr=True, eq=True, order=False, unsafe_hash=False, frozen=False)
````

**File:** `~/aai-ws/fastcore/fastcore/xtras.py`; line: 1003

In [ ]:
def f(a:int=0 # aa
): pass

@delegates(f)
def g(
    b:int, # bb
    **kwargs
)->int: # Returns the meaning of life
    "The g function"
    # nothing to see here
    pass

In [ ]:
g?

````python
def g(
    b:int, # bb
    a:int=0, # aa
)->int: # Returns the meaning of life
    "The g function"
````

**File:** `/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipymini_57881/3905198878.py`; line: 4

**Type:** function

In [ ]:
g??

````python
@delegates(f)
def g(
    b:int, # bb
    **kwargs
)->int: # Returns the meaning of life
    "The g function"
    # nothing to see here
    pass
````

**File:** `/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipymini_57881/3905198878.py`; line: 4

In [ ]:
def info_md(
    obj:str|object, # Object to get info for, or its name
    source:bool=False
):
    "With `source=False` same as ipython's `?`, otherwise same as `??`"
    obj = resolve(obj)
    r = get_ipython().inspector._get_info(obj, '', detail_level=1 if source else 0)
    return PrettyString(r.get('text/markdown') or re.sub(r'\x1b\[[0-9;]*m', '', r['text/plain']))

In [ ]:
info_md(flexiclass)

````python
def flexiclass(
    cls, # The class to convert
)->dataclass:
    "Convert `cls` into a `dataclass` like `make_nullable`. Converts in place and also returns the result."
````

**File:** `~/aai-ws/fastcore/fastcore/xtras.py`; line: 1003

**Type:** function

In [ ]:
res = info_md(flexiclass)
assert re.search(r':\n    "', res)  # docstring indented as function body
assert f'````\n{flexiclass.__doc__}\n````' not in res  # not repeated as a separate block


`info_md` also works on instances (reporting their type's info), builtins, and other objects with no retrievable source:

In [ ]:
res = info_md(Path('.'))
assert 'PosixPath' in res and 'pathlib' in res
assert 'class PosixPath' in info_md(Path('.'), source=True)
Markdown(res)

<div class="prose" markdown="1">

````python
PosixPath instance: Path('.')
    "Path subclass for non-Windows systems.

On a POSIX system, instantiating a Path should return this object."
````

**String form:** `.`

**File:** `~/.local/share/uv/python/cpython-3.13-macos-aarch64-none/lib/python3.13/pathlib/_local.py`; line: 839

**Type:** PosixPath

</div>

In [ ]:
Markdown(info_md(len))

<div class="prose" markdown="1">

````python
def len(
    obj
):
    "Return the number of items in a container."
````

**Type:** builtin_function_or_method

</div>

Workaround for error where non str evalue causes a crash:

In [ ]:
from IPython.core.ultratb import SyntaxTB

In [ ]:
@patch()
def structured_traceback(self:SyntaxTB, etype, evalue, etb, tb_offset=None, context=5):
    if hasattr(evalue, 'msg') and not isinstance(evalue.msg, str): evalue.msg = str(evalue.msg)
    return self._orig_structured_traceback(etype, evalue, etb, tb_offset=tb_offset, context=context)

Workaround for error where non str `__file__` causes a crash:

In [ ]:
@patch_to(inspect, nm="getfile")
def _getfile(obj): return str(inspect._orig_getfile(obj))

In [ ]:
def _fmt_magic_res(result): return HTML(to_xml(result)) if isinstance(result, FT) else result

def load_ipython_extension(ip):
    ns = ip.user_ns
    for o in ('read_url','transient','run_cmd','maybe_await','call_tool'): ns[o] = globals()[o]
    enable_async_magics(ip, fmt=_fmt_magic_res)  # magics (line and cell) may be async; FT results become HTML

    dh_cls = type(ip.displayhook)
    _orig_write_format_data = dh_cls.write_format_data
    def write_format_data(self, format_dict, md_dict=None):
        obj = self.shell.user_ns.get('_')
        if obj is not None:
            if md_dict is None: md_dict = {}
            md_dict['__type'] = type(obj).__qualname__
        return _orig_write_format_data(self, format_dict, md_dict)
    dh_cls.write_format_data = write_format_data


In [ ]:
ip = get_ipython()
load_ipython_extension(ip)
r = ip.run_cell_magic('time', '', 'pass')
test_eq(inspect.iscoroutine(r), False)
assert 'await' in ip.input_transformer_manager.transform_cell('%time pass')  # magics now awaitable via fastcore.aio.enable_async_magics

## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()